Воспроизведем исходный сигнал

# Лабораторная работа №1
## *Частотный конвертер*
по курсу Цифровая обработка сигналов  
  
**направление:** Речевые технологии и машинное обучение  
**преподаватель:** Рыбин Сергей Витальевич  
**выполнили:** Иванова Мария Кирилловна  
**группа:** М4121

### Импортируем необходимые библиотеки

In [1]:
import librosa
import numpy as np
from math import gcd
import matplotlib.pyplot as plt
import IPython.display as ipd
from scipy.signal import firwin, lfilter

### Загрузим и воспроизведем исходный сигнал

In [ ]:
signal, sr = librosa.load('data/.', sr=None)
print("Исходная частота дискретизации:", sr)
ipd.Audio(signal, rate=sr)

### Оставим 10 секунд для удобства

In [ ]:
signal, sr = librosa.load('data/.', offset=91.0, duration=10.0, sr=44100)
ipd.Audio(signal, rate=sr)

### Реализуем частотный конвертер для двух случаев

### $$ N_1 > N_2 $$

In [4]:
N1 = 44100  # исходная частота
N2 = 8000   # новая частота

# НОД для ускорения работы алгоритма
g = gcd(N1, N2)
L = N2 // g
M = N1 // g

# длина сигнала после растяжения
len_sig = len(signal) * L
x_hat = np.zeros(len_sig)

# прореживание
for j in range(len_sig):
    if j % L == 0:
        x_hat[j] = signal[j // L]
    else:
        x_hat[j] = 0.0

print(f"Размер исходного сигнала: {len(signal)}")
print(f"Размер растянутого сигнала: {len(x_hat)}")        

# Фильтрация с КИХ фильтром
Len = 300
fs = N1 * L          
cutoff = N2 / 2     

hK = firwin(Len, cutoff=cutoff, window=('kaiser', 9), fs=fs)
y_hat = lfilter(hK, 1.0, x_hat)

# Прореживание
y = y_hat[::M]

# Воспроизведение получившегося сигнала
print(f"Новая частота дискретизации: {N2} Гц")
ipd.Audio(y, rate=N2)

Размер исходного сигнала: 441000
Размер растянутого сигнала: 35280000
Новая частота дискретизации: 8000 Гц


### Сравним результаты построенного вручную конвертера и встроенного ресемплинга библиотеки librosa

In [5]:
# Ресемплинг с помощью librosa
y_librosa = librosa.resample(signal, orig_sr=N1, target_sr=N2, res_type='kaiser_best')

# Сравнение сигналов
# Нормируем, чтобы сравнение было по форме
min_len = min(len(y), len(y_librosa))
y = y[:min_len]
y_librosa = y_librosa[:min_len]

# Вычислим разницу
diff = np.mean(np.abs(y - y_librosa))
print(f"Средняя разница между сигналами: {diff:.6f}")

# Воспроизведение
print("Ручной ресемплинг:")
display(ipd.Audio(y, rate=N2))
print("Librosa ресемплинг:")
display(ipd.Audio(y_librosa, rate=N2))

Средняя разница между сигналами: 0.052024
Ручной ресемплинг:


Librosa ресемплинг:


### $$ N_1 < N_2 $$

### 10-секундный сигнал с меньшей частотой дискретизации

In [ ]:
signal, sr = librosa.load('data/.', offset=91.0, duration=10.0, sr=4000)
ipd.Audio(signal, rate=sr)

### Частотный конвертер

In [7]:
N1 = 4000    
N2 = 16000  

g = gcd(N1, N2)
L = N2 // g   
M = N1 // g   

# Прореживание
len_sig = len(signal) * L
x_hat = np.zeros(len_sig, dtype=signal.dtype)

for j in range(len_sig):
    if j % L == 0:
        x_hat[j] = signal[j // L]
    else: x_hat[j] = 0.0

print(f"Размер исходного сигнала: {len(signal)}")
print(f"Размер растянутого сигнала: {len(x_hat)}")    

# КИХ фильтр
Len = 1001                        
fs_up = N1 * L                   
cutoff_hz = 0.5 * min(N1, N2)    
hK = firwin(numtaps=Len, cutoff=cutoff_hz, fs=fs_up, window=('kaiser', 9))

y_hat = lfilter(hK, 1.0, x_hat)

# ПрореПживание
y = y_hat[::M]

print(f"Новая частота дискретизации: {N2} Гц")
ipd.Audio(y, rate=N2)


Размер исходного сигнала: 40000
Размер растянутого сигнала: 160000
Новая частота дискретизации: 16000 Гц


### Сравнение с ресемплингом librosa

In [8]:
# Ресемплинг с помощью librosa
y_librosa = librosa.resample(signal, orig_sr=N1, target_sr=N2, res_type='kaiser_best')

# Сравнение сигналов
# Нормировка
min_len = min(len(y), len(y_librosa))
y = y[:min_len]
y_librosa = y_librosa[:min_len]

# Вычислим разницу
diff = np.mean(np.abs(y - y_librosa))
print(f"Средняя разница между сигналами: {diff:.6f}")

# Воспроизведение
print("Ручной ресемплинг:")
display(ipd.Audio(y, rate=N2))
print("Librosa ресемплинг:")
display(ipd.Audio(y_librosa, rate=N2))

Средняя разница между сигналами: 0.051431
Ручной ресемплинг:


Librosa ресемплинг:


## Выводы  
- #### При изменении частоты дискретизации с помощью конвертера слышны изменения в звуке. Он становится либо чище (для случая N1 < N2), либо более тусклым и менее насыщенным (для случая N1 > N2)
- #### Реализованные вручную частотные конверторы несильно отличаются от библиотечных. Погрешность составляет 5%
- #### На слух различить два варианта конвертации практически невозможно